# Task 6 — The Binary Decision
**PlaceMux AI/ML Phase 1** | Binary classifier + confusion matrix + precision/recall + justified threshold

In [2]:
import sys, os; sys.path.insert(0, '..')
import random, numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib
matplotlib.use('Agg')
from IPython.display import Image, display
from src.preprocessing import load_and_split, build_preprocessor
from src.train import train_model, save_model
from src.evaluate import full_eval, plot_confusion_matrix, plot_roc_pr, threshold_analysis
from src.predict import Predictor
import joblib
SEED=42; random.seed(SEED); np.random.seed(SEED)
print('✅ Setup complete')

ImportError: cannot import name 'Predictor' from 'src.predict' (C:\Users\vedan\PycharmProjects\AI-ML\Phase 1 Task 6 - The Binary Decision\notebooks\..\src\predict.py)

## 1 — Dataset & Class Imbalance

In [ ]:
df = pd.read_csv('../data/raw/credit_fraud_dataset.csv')
print('Shape:', df.shape, '| Nulls:', df.isnull().sum().sum())
print('\nClass Distribution:')
print(df['is_fraud'].value_counts())
print('\nFraud rate:', df['is_fraud'].mean().round(3))

fig, ax = plt.subplots(figsize=(6,3))
df['is_fraud'].value_counts().plot(kind='bar', ax=ax, color=['steelblue','tomato'], edgecolor='black')
ax.set_xticklabels(['Not Fraud','Fraud'], rotation=0)
ax.set_title('Class Distribution (Imbalanced: 79% vs 21%)')
plt.tight_layout(); plt.savefig('../outputs/class_dist.png', dpi=100); plt.show()
print('⚠ Accuracy alone is misleading — a model predicting Not Fraud always gets 79% accuracy!')

## 2 — Load, Split, Preprocess

In [ ]:
X_tr,X_val,X_test,y_tr,y_val,y_test = load_and_split('../data/raw/credit_fraud_dataset.csv', seed=SEED)
pp, X_tr_p = build_preprocessor(X_tr)
X_val_p = pp.transform(X_val)
X_test_p = pp.transform(X_test)

## 3 — Train Both Models

In [ ]:
lr = train_model('logistic', X_tr_p, y_tr, SEED)
rf = train_model('random_forest', X_tr_p, y_tr, SEED, {'n_estimators':100})

## 4 — Evaluate @ Default Threshold 0.5

In [ ]:
lr_m, lr_proba, lr_pred = full_eval(lr, X_val_p, y_val, 'val', 'logistic', 0.5)
rf_m, rf_proba, rf_pred = full_eval(rf, X_val_p, y_val, 'val', 'random_forest', 0.5)
pd.DataFrame([lr_m, rf_m])[['model','accuracy','precision','recall','f1','roc_auc','pr_auc']]

## 5 — ROC & Precision-Recall Curves

In [ ]:
plot_roc_pr([('Logistic',y_val,lr_proba),('RandomForest',y_val,rf_proba)], '../outputs')
display(Image('../outputs/roc_pr_curves.png'))

## 6 — Threshold Analysis (validation set)

In [ ]:
thr_df = threshold_analysis(y_val, rf_proba, out_path='../outputs/threshold_analysis.csv')
display(Image('../outputs/threshold_analysis.png'))
thr_df

## 7 — Business Cost Analysis & Threshold Selection

In [ ]:
print('Business Cost Analysis:')
print('  False Negative (missed fraud) → financial loss, fraud goes undetected → HIGH COST')
print('  False Positive (false alarm)  → customer inconvenience, extra verification → LOW COST')
print('\nDecision: Prioritise RECALL >= 0.70 to minimise missed frauds')
print()
candidates = thr_df[thr_df['recall'] >= 0.70]
best = candidates.loc[candidates['f1'].idxmax()]
FINAL_THR = float(best['threshold'])
print(f'Optimal threshold = {FINAL_THR}')
print(best)

## 8 — Final Test Evaluation (once, unseen data)

In [ ]:
test_m, test_proba, test_pred = full_eval(rf, X_test_p, y_test, 'test', 'random_forest', FINAL_THR)
plot_confusion_matrix(y_test, test_pred, f'Final Test — threshold={FINAL_THR}', '../outputs/cm_final_test.png')
display(Image('../outputs/cm_final_test.png'))

## 9 — Live Prediction Demo

In [ ]:
p = Predictor('../models/random_forest.pkl', '../models/preprocessor.pkl', threshold=FINAL_THR)

high_risk = {'age':34,'income':72000,'credit_limit':18000,'transaction_amount':4200,
             'num_transactions_30d':55,'account_age_months':8,'num_prev_disputes':3,
             'merchant_category':'electronics','country_match':0,'time_of_day_hour':3,
             'is_weekend':1,'card_present':0,'distance_from_home_km':380}
low_risk = {'age':45,'income':90000,'credit_limit':25000,'transaction_amount':85,
             'num_transactions_30d':12,'account_age_months':120,'num_prev_disputes':0,
             'merchant_category':'food','country_match':1,'time_of_day_hour':13,
             'is_weekend':0,'card_present':1,'distance_from_home_km':5}

print('HIGH RISK:', p.predict(high_risk))
print('LOW RISK: ', p.predict(low_risk))

## 10 — Edge Case Handling

In [ ]:
# Missing values — handled by pipeline
missing = {**low_risk, 'age': None, 'distance_from_home_km': None}
print('Missing values:', p.predict(missing))

# Unseen category
unseen = {**low_risk, 'merchant_category': 'supermarket'}
print('Unseen category:', p.predict(unseen))

# Invalid input
try:
    p.predict({**low_risk, 'age':'hello', 'income':-999})
except ValueError as e:
    print('Invalid input caught:', e)

## ✅ Task 6 Summary

| Metric | Default thr=0.5 | **Business thr=0.25** |
|---|---|---|
| Accuracy | 80.0% | 73.7% |
| Precision | 55.0% | 42.1% |
| **Recall** | 17.7% | **72.6%** |
| F1 | 0.268 | 0.533 |
| ROC-AUC | 0.832 | 0.832 |
| FN (missed fraud) | 51 | **17** |

**Justification:** At threshold=0.25, we catch 72.6% of all frauds (vs 17.7% at default 0.5). We accept more false alarms because the cost of missing real fraud far exceeds the cost of an extra verification check.